<a href="https://colab.research.google.com/github/idialloaka-ai/DAILYCHALLENGE/blob/master/Daily_challenge_W7D2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Défi Pinecone : Reclassement sans serveur en action
Ce notebook contient la solution complète pour le challenge Pinecone.

In [ ]:
!pip install -U pinecone==6.0.1 pinecone-notebooks pandas torch transformers requests

In [ ]:
import os
if not os.environ.get("PINECONE_API_KEY"):
    from pinecone_notebooks.colab import Authenticate
    Authenticate()

from pinecone import Pinecone
api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=api_key)

### Partie 1 : Reranking de base

In [ ]:
query = "Tell me about Apple's products"
documents = [
    "The Granny Smith is a tip-bearing apple cultivar, which originated in Australia in 1868.",
    "The iPhone 15 Pro features a strong and lightweight aerospace-grade titanium design.",
    "Apples are high in fiber and vitamin C, and they are also low in calories.",
    "The MacBook Air with M3 chip provides incredible performance and up to 18 hours of battery life.",
    "The Apple Watch Ultra 2 is the most capable and ruggedized smartwatch from the company."
]

from pinecone import RerankModel
reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=query,
    documents=[{"id": str(i), "text": doc} for i, doc in enumerate(documents)],
    top_n=3
)

def show_reranked_results(query, rerank_response):
    print(f"Query: {query}")
    for i, m in enumerate(rerank_response.data):
        print(f"{i+1}. Score: {m.score:.4f} | Text: {m.document.text}")

show_reranked_results(query, reranked)

### Partie 2 & 3 : Index sans serveur et chargement des données

In [ ]:
from pinecone import ServerlessSpec
import time
import requests
import tempfile
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel

cloud = os.getenv('PINECONE_CLOUD', 'aws')
region = os.getenv('PINECONE_REGION', 'us-east-1')
spec = ServerlessSpec(cloud=cloud, region=region)
index_name = 'medical-notes-index'

if pc.has_index(name=index_name):
    pc.delete_index(name=index_name)

pc.create_index(
    name=index_name,
    dimension=384,
    metric='cosine',
    spec=spec
)

with tempfile.TemporaryDirectory() as tmpdirname:
    file_path = os.path.join(tmpdirname, "sample_notes_data.jsonl")
    url = "https://raw.githubusercontent.com/pinecone-io/examples/refs/heads/master/docs/data/sample_notes_data.jsonl"
    response = requests.get(url)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)
    df = pd.read_json(file_path, orient='records', lines=True)

print("Data shape:", df.shape)
display(df.head())

### Partie 4 & 5 : Upsert et Recherche Sémantique

In [ ]:
index = pc.Index(name=index_name)
index.upsert_from_dataframe(df)

def is_fresh(index):
    stats = index.describe_index_stats()
    return stats.total_vector_count > 0

print("Waiting for index availability...")
while not is_fresh(index):
    time.sleep(5)
print("Index ready!")

def get_embedding(input_question):
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    encoded_input = tokenizer(input_question, padding=True, truncation=True, return_tensors='pt')
    with torch.no_grad():
        model_output = model(**encoded_input)
        embedding = model_output.last_hidden_state[0].mean(dim=0)
    return embedding

question = "patient with chest pain and breathing issues"
query_vec = get_embedding(question).tolist()
results = index.query(vector=[query_vec], top_k=5, include_metadata=True)
sorted_matches = sorted(results['matches'], key=lambda x: x['score'], reverse=True)

### Partie 6 : Affichage et Reclassement Final

In [ ]:
def show_results(question, matches):
    print(f'Question: \'{question}\'\nResults:')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match["id"]}')
        print(f' Score: {match["score"]}')
        print(f' Metadata: {match["metadata"]}\n')

show_results(question, sorted_matches)

transformed_documents = [
    {
        'id': match['id'],
        'reranking_field': '; '.join([f"{key}: {value}" for key, value in match['metadata'].items()])
    }
    for match in results['matches']
]

refined_query = "Is there a plan for managing the patient's acute chest pain and potential fracture?"
reranked_results = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=refined_query,
    documents=transformed_documents,
    rank_fields=["reranking_field"],
    top_n=3,
    return_documents=True,
)

def show_reranked_results_final(question, response):
    print(f'Refined Question: \'{question}\'\nReranked Results:')
    for i, match in enumerate(response.data):
        print(f'{str(i+1).rjust(4)}. ID: {match.document.id}')
        print(f' Score: {match.score}')
        print(f' Reranking Field: {match.document.reranking_field}\n')

show_reranked_results_final(refined_query, reranked_results)

### Nettoyage

In [ ]:
# pc.delete_index(name=index_name)
print("Index maintenu pour vérification. Décommentez la ligne ci-dessus pour supprimer.")